# workspace-agent-gym — GPU model server on Colab

Serves a Qwen2.5 model from a Colab GPU through a public tunnel, so the gym
running on your local machine (Mattermost in Docker + the harness) can use it
as its Ollama backend — **zero code changes**, the `OLLAMA_URL` env var is
already supported.

**Runtime → Change runtime type → GPU.** Pick `MODEL` in cell 2 to match your
GPU (the cell prints VRAM guidance).

Architecture: the environment and graders stay local (Docker isn't available
on Colab); only inference happens here.

```
[your laptop]  run_eval + Mattermost(Docker)  --HTTPS-->  [Colab GPU]  Ollama + Qwen2.5
```

In [ ]:
# 1. Install zstd (needed by the Ollama installer), Ollama, and cloudflared
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL -o /usr/local/bin/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Pick the model for your GPU, start the server, pull the model
#    VRAM guide (Q4 quants):  qwen2.5:7b ~5 GB (T4 ok)
#                             qwen2.5:32b ~20 GB (A100/L4 40GB+, RTX 6000...)
#                             qwen2.5:72b ~47 GB (80GB+ cards)
MODEL = 'qwen2.5:32b'   # <-- change to match your GPU

import subprocess, time, os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
server = subprocess.Popen(['ollama', 'serve'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull {MODEL}
!ollama list

In [ ]:
# 3. Open the tunnel — copy the printed URL and commands
import re, subprocess, time
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434',
     '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
slug = MODEL.replace(':', '_')
print('\n' + '=' * 70)
print('OLLAMA_URL =', url)
print('=' * 70)
print('\nOn your local machine (repo root, Mattermost already up) — PowerShell:\n')
print(f'  $env:OLLAMA_URL = "{url}"')
print(f'  python -m scripts.run_eval --backend ollama:{MODEL}')
print(f'  python -m calibration.report results/ollama_{slug}.jsonl')

In [ ]:
# 4. Smoke test the served model through the tunnel
import requests
r = requests.post(f'{url}/api/chat', json={
    'model': MODEL, 'stream': False,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: ready'}]})
print(r.json()['message']['content'])
# Keep this notebook running while the eval executes locally.
# If Colab disconnects mid-eval, rerun cells 2-4 (the tunnel URL will change)
# and relaunch run_eval — use --limit N to work in chunks.